# Combined RNA + Epigenetic Discordance Analysis
## Identifying TF/ncRNA-driven gene regulation in Layer V ET

**Discordance Score (WDS):**
```
WDS = α·Z(CGN) + β·Z(CHN) - Z(RNA)
```
- `WDS >> 0` → High methylation + Low expression → **FORCED_SUPPRESSION** (TF/ncRNA suppressing despite open-ish chromatin... wait, or blocking despite closed)
- `WDS << 0` → Low methylation + High expression → **FORCED_EXPRESSION** (TF/ncRNA forcing expression through closed chromatin)

**Narrowing logic:**
- Discordant in **all** cell types → not identity-specific
- Discordant only in **neurons** → neuron-identity regulation
- Discordant only in **exitory neurons** → exitory-neuron-identity regulation
- Discordant only in **deep layer neurons** → deep layer regulation
- Discordant only in **Layer V ET** → Layer V ET-specific regulation

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import zscore

sc.settings.verbosity = 1
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)

/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: 

## 0. Configuration

In [2]:
# ── EDIT THESE PATHS ─────────────────────────────────────────────────────────
PATH_RNA = "/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad"
PATH_CGN = "/home/nakagawa/datasets/h5ad/DNA_Methylation_CGN.h5ad"
PATH_CHN = "/home/nakagawa/datasets/h5ad/DNA_Methylation_CHN.h5ad"
OUT_DIR  = Path("/home/nakagawa/datasets/discordance_results")
# ─────────────────────────────────────────────────────────────────────────────

# Weighted Discordance Score weights (must sum to 1.0)
ALPHA = 0.6   # CGN weight (canonical CpG silencing)
BETA  = 0.4   # CHN weight (neuron-specific non-CpG silencing)

# Discordance thresholds
DS_THRESH = 2.0    # |WDS| >= this to call discordant (in Z-score units)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration set.")

Configuration set.


## 1. Load All Three Datasets

In [3]:
print("Loading RNA...")
adata_rna = sc.read_h5ad(PATH_RNA)
print(f"  RNA : {adata_rna.shape[0]} cells x {adata_rna.shape[1]} genes")

print("Loading CGN methylation...")
adata_cgn = sc.read_h5ad(PATH_CGN)
print(f"  CGN : {adata_cgn.shape[0]} cells x {adata_cgn.shape[1]} genes")

print("Loading CHN methylation...")
adata_chn = sc.read_h5ad(PATH_CHN)
print(f"  CHN : {adata_chn.shape[0]} cells x {adata_chn.shape[1]} genes")

Loading RNA...
  RNA : 71183 cells x 30198 genes
Loading CGN methylation...
  CGN : 9876 cells x 51574 genes
Loading CHN methylation...
  CHN : 9876 cells x 51573 genes


## 2. Check Annotations Are Consistent Across Files

In [4]:
print("=" * 60)
print("RNA metadata columns:")
print(adata_rna.obs.columns.tolist())

print("\nCGN metadata columns:")
print(adata_cgn.obs.columns.tolist())

print("\nCHN metadata columns:")
print(adata_chn.obs.columns.tolist())

RNA metadata columns:
['aggr_num', 'umi.counts', 'gene.counts', 'library_id', 'tube_barcode', 'Seq_batch', 'Region', 'Lib_type', 'donor_id', 'Amp_Name', 'Amp_Date', 'Amp_PCR_cyles', 'Lib_Date', 'Replicate_Lib', 'Lib_PCR_cycles', 'Lib_PassFail', 'Cell_Capture', 'Lib_Cells', 'Mean_Reads_perCell', 'Median_Genes_perCell', 'Median_UMI_perCell', 'Saturation', 'Live_percent', 'Total_Cells', 'Live_Cells', 'exp_component_name', 'mapped_reads', 'unmapped_reads', 'nonconf_mapped_reads', 'total.reads', 'doublet.score', 'row', 'BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'temp_class_label', 'BICCN_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_report

In [10]:
# ── EDIT: set the cell type column name for each file ────────────────────────
CT_COL_RNA = "BICCN_subclass_label"
CT_COL_CGN = "BICCN_cluster_label"
CT_COL_CHN = "BICCN_cluster_label"
# ─────────────────────────────────────────────────────────────────────────────

# Map methylation cluster labels → RNA subclass labels so the sets can overlap
meth_to_subclass = {
    "L23-IT-Cux2":              "L2/3 IT",
    "L4-IT-Rorb_Rorb-Tenm2":   "L4 IT",
    "L4-IT-Rorb_Rorb-Cpne4":   "L4 IT",
    "L4-IT-Rorb_Rorb-Ryr3":    "L4 IT",
    "L5-IT-Deptor":             "L5 IT",
    "L5-PT-Bcl6":               "L5 ET",   # PT (pyramidal tract) = ET
    "L6-CT-Foxp2_Foxp2-Kcnh5": "L6 CT",
    "L6-CT-Foxp2_Foxp2-Glra2": "L6 CT",
    "L6-CT-Foxp2_Foxp2-Wscd1": "L6 CT",
    "L6-CT-Foxp2_Spon1":        "L6 CT",
    "L6-CT-Foxp2_Hcrtr2":       "L6 CT",
    "L6-NP-Tshz2":              "L5/6 NP",
    "L6-IT-Sulf1_Cables1":      "L6 IT",
    "L6-IT-Sulf1_Meis2":        "L6 IT",
    "L6-IT-Sulf1_Sulf1":        "L6 IT",
    "L6-IT-Sulf1_Pcdh15":       "L6 IT",
    "L6b-Galnt10":              "L6b",
    "MGE-Pvalb_Chrna7":         "Pvalb",
    "MGE-Pvalb_Cnih3":          "Pvalb",
    "MGE-Pvalb_Man2a1":         "Pvalb",
    "MGE-Pvalb_Unc5b":          "Pvalb",
    "MGE-Sst_Gfra2":            "Sst",
    "MGE-Sst_Daam2":            "Sst",
    "MGE-Sst_Srrm4":            "Sst",
    "MGE-Sst_Prep":             "Sst",
    "MGE-Sst_Kcnip4":           "Sst",
    "MGE-Sst_Whrn":             "Sst",
    "MGE-Sst_Chodl":            "Sst",
    "CGE-VipNdnf_Lamp5-Unc5d":  "Lamp5",
    "CGE-VipNdnf_Lamp5-Ndnf":   "Lamp5",
    "CGE-VipNdnf_Vip-Ano4":     "Vip",
    "CGE-VipNdnf_Vip-Prr16":    "Vip",
    "CGE-VipNdnf_Vip-Unc5b":    "Vip",
    "CGE-VipNdnf_Vip-Sorcs1":   "Vip",
    "CGE-VipNdnf_Pex5l":        "Sncg",
    "NonN_ODC-Mog":             "Oligo",
    "NonN_Astro-Slc1a2":        "Astro",
    "NonN_Micro-Csf1r":         "Micro",
    "NonN_VLMC-Slc6a13":        "VLMC",
    "NonN_Endo-Nxn":            "Endo",
    "NonN_OPC-Pdgfra":          "OPC",
    "NonN_SMC-Myo1b":           "SMC",
    "NonN_Micro-Csf1r":         "Macrophage",
    # Outlier intentionally omitted → maps to NaN → dropped below
}

CT_COL_METH = "subclass_label"   # new harmonised column name

for adata_m in [adata_cgn, adata_chn]:
    adata_m.obs[CT_COL_METH] = adata_m.obs[CT_COL_CGN].map(meth_to_subclass)

# Drop unmapped rows (Outlier, anything not in the dict)
adata_cgn = adata_cgn[adata_cgn.obs[CT_COL_METH].notna()].copy()
adata_chn = adata_chn[adata_chn.obs[CT_COL_METH].notna()].copy()

CT_COL_CGN = CT_COL_METH
CT_COL_CHN = CT_COL_METH

# ── Now the set operations work ───────────────────────────────────────────────
ct_rna = set(adata_rna.obs[CT_COL_RNA].unique())
ct_cgn = set(adata_cgn.obs[CT_COL_CGN].unique())
ct_chn = set(adata_chn.obs[CT_COL_CHN].unique())

print(f"Cell types in RNA : {len(ct_rna)}")
print(f"Cell types in CGN : {len(ct_cgn)}")
print(f"Cell types in CHN : {len(ct_chn)}")

shared_ct = ct_rna & ct_cgn & ct_chn
print(f"\nShared across all three: {len(shared_ct)}")

only_rna = ct_rna - ct_cgn - ct_chn
only_cgn = ct_cgn - ct_rna - ct_chn
only_chn = ct_chn - ct_rna - ct_cgn

if only_rna: print(f"\n[WARNING] Only in RNA : {only_rna}")
if only_cgn: print(f"[WARNING] Only in CGN : {only_cgn}")
if only_chn: print(f"[WARNING] Only in CHN : {only_chn}")

print("\nShared cell types:")
for ct in sorted(shared_ct):
    n_rna = (adata_rna.obs[CT_COL_RNA] == ct).sum()
    n_cgn = (adata_cgn.obs[CT_COL_CGN] == ct).sum()
    n_chn = (adata_chn.obs[CT_COL_CHN] == ct).sum()
    print(f"  {ct:<30} RNA={n_rna:>5}  CGN={n_cgn:>5}  CHN={n_chn:>5}")

Cell types in RNA : 20
Cell types in CGN : 20
Cell types in CHN : 20

Shared across all three: 19

[WARNING] Only in RNA : {'L6 IT Car3'}

Shared cell types:
  Astro                          RNA=  398  CGN=  109  CHN=  109
  Endo                           RNA=  187  CGN=   30  CHN=   30
  L2/3 IT                        RNA=10915  CGN= 1965  CHN= 1965
  L5 ET                          RNA=  161  CGN=  354  CHN=  354
  L5 IT                          RNA=29721  CGN=  898  CHN=  898
  L5/6 NP                        RNA= 3147  CGN=  275  CHN=  275
  L6 CT                          RNA=12807  CGN= 1580  CHN= 1580
  L6 IT                          RNA= 4445  CGN=  873  CHN=  873
  L6b                            RNA=  554  CGN=  141  CHN=  141
  Lamp5                          RNA= 2357  CGN=  292  CHN=  292
  Macrophage                     RNA=  122  CGN=   68  CHN=   68
  OPC                            RNA=  146  CGN=   28  CHN=   28
  Oligo                          RNA=  537  CGN=  144  CHN=  1

In [11]:
# Check gene name overlap
genes_rna = set(adata_rna.var_names)
genes_cgn = set(adata_cgn.var_names)
genes_chn = set(adata_chn.var_names)

shared_genes = genes_rna & genes_cgn & genes_chn
print(f"Genes in RNA       : {len(genes_rna)}")
print(f"Genes in CGN       : {len(genes_cgn)}")
print(f"Genes in CHN       : {len(genes_chn)}")
print(f"Shared across all  : {len(shared_genes)}")

# Sample check — first 10 gene names per file
print("\nRNA gene name examples :", list(adata_rna.var_names[:5]))
print("CGN gene name examples :", list(adata_cgn.var_names[:5]))
print("CHN gene name examples :", list(adata_chn.var_names[:5]))

Genes in RNA       : 30198
Genes in CGN       : 51574
Genes in CHN       : 51573
Shared across all  : 30147

RNA gene name examples : ['ENSMUSG00000029422', 'ENSMUSG00000114536', 'ENSMUSG00000049036', 'ENSMUSG00000029577', 'ENSMUSG00000040746']
CGN gene name examples : ['ENSMUSG00000050732', 'ENSMUSG00000101535', 'ENSMUSG00000101517', 'ENSMUSG00000103873', 'ENSMUSG00000099414']
CHN gene name examples : ['ENSMUSG00000050732', 'ENSMUSG00000101535', 'ENSMUSG00000101517', 'ENSMUSG00000103873', 'ENSMUSG00000099414']


## 3. Define Cell Type Groups
Fill in after confirming shared cell types above.

In [12]:
# ── EDIT: use exact strings from the shared cell type list above ──────────────
LAYER_V_ET = "L5 ET"

NON_NEURON_TYPES = [
    "Astro",
    "Macrophage",
    "Oligo",
    "OPC",
    "Endo",
    "VLMC",
    "SMC",
]

UPPER_LAYER_TYPES = [
    "L2/3 IT",
]

OTHER_DEEP_LAYER_TYPES = [
    "L5 IT",
    "L5/6 NP",
    "L6 IT",
    "L6 CT",
    "L6b",
]

INHIBITORY_TYPES = [
    "Pvalb",
    "Sst",
    "Vip",
    "Lamp5",
    "Sncg",
]
# ─────────────────────────────────────────────────────────────────────────────

ALL_CELL_TYPES = (
    [LAYER_V_ET]
    + NON_NEURON_TYPES
    + UPPER_LAYER_TYPES
    + OTHER_DEEP_LAYER_TYPES
    + INHIBITORY_TYPES
)

# Filter to only those actually present in all three datasets
ALL_CELL_TYPES = [ct for ct in ALL_CELL_TYPES if ct in shared_ct]
print(f"Working with {len(ALL_CELL_TYPES)} cell types present in all three datasets:")
for ct in ALL_CELL_TYPES:
    print(f"  {ct}")

Working with 19 cell types present in all three datasets:
  L5 ET
  Astro
  Macrophage
  Oligo
  OPC
  Endo
  VLMC
  SMC
  L2/3 IT
  L5 IT
  L5/6 NP
  L6 IT
  L6 CT
  L6b
  Pvalb
  Sst
  Vip
  Lamp5
  Sncg


## 4. Compute Pseudobulk Mean Per Cell Type
We average across cells per cell type — this gives one value per gene per cell type,
which is what we Z-score across cell types.

In [13]:
def pseudobulk_mean(adata, celltype_col, cell_types, gene_set):
    """
    Returns DataFrame: rows=genes (shared_genes), cols=cell_types
    Values = mean expression/methylation per cell type.
    """
    gene_list = sorted(gene_set & set(adata.var_names))
    result = {}
    for ct in cell_types:
        mask = adata.obs[celltype_col] == ct
        sub  = adata[mask][:, gene_list]
        if hasattr(sub.X, 'toarray'):
            vals = sub.X.toarray()
        else:
            vals = np.array(sub.X)
        result[ct] = vals.mean(axis=0)
    return pd.DataFrame(result, index=gene_list)   # genes x cell_types


print("Computing pseudobulk means...")

# RNA: normalize first if raw counts
adata_rna_use = adata_rna.raw.to_adata() if adata_rna.raw else adata_rna.copy()
adata_rna_use.obs[CT_COL_RNA] = adata_rna.obs[CT_COL_RNA]
if adata_rna_use.X.max() > 100:
    sc.pp.normalize_total(adata_rna_use, target_sum=1e4)
    sc.pp.log1p(adata_rna_use)
    print("  RNA: normalized + log1p")

pb_rna = pseudobulk_mean(adata_rna_use, CT_COL_RNA, ALL_CELL_TYPES, shared_genes)
print(f"  RNA pseudobulk : {pb_rna.shape}")

# CGN/CHN: methylation fractions, no normalization needed
pb_cgn = pseudobulk_mean(adata_cgn, CT_COL_CGN, ALL_CELL_TYPES, shared_genes)
print(f"  CGN pseudobulk : {pb_cgn.shape}")

pb_chn = pseudobulk_mean(adata_chn, CT_COL_CHN, ALL_CELL_TYPES, shared_genes)
print(f"  CHN pseudobulk : {pb_chn.shape}")

# Align gene index
common_idx = pb_rna.index.intersection(pb_cgn.index).intersection(pb_chn.index)
pb_rna = pb_rna.loc[common_idx]
pb_cgn = pb_cgn.loc[common_idx]
pb_chn = pb_chn.loc[common_idx]
print(f"\nFinal gene set for analysis: {len(common_idx)} genes")

Computing pseudobulk means...
  RNA: normalized + log1p
  RNA pseudobulk : (30147, 19)
  CGN pseudobulk : (30147, 19)
  CHN pseudobulk : (30147, 19)

Final gene set for analysis: 30147 genes


## 5. Z-score Normalize Each Matrix
Z-scored **across cell types** per gene — captures relative deviation from the mean cell type behaviour.

In [14]:
def zscore_across_celltypes(df):
    """
    Z-score each gene (row) across cell types (columns).
    Genes with zero variance across cell types get DS=0 (they're not informative).
    """
    arr    = df.values.astype(float)
    means  = arr.mean(axis=1, keepdims=True)
    stds   = arr.std(axis=1, keepdims=True)
    stds[stds == 0] = 1   # avoid division by zero
    z      = (arr - means) / stds
    return pd.DataFrame(z, index=df.index, columns=df.columns)


z_rna = zscore_across_celltypes(pb_rna)
z_cgn = zscore_across_celltypes(pb_cgn)
z_chn = zscore_across_celltypes(pb_chn)

print("Z-score ranges (should be centred near 0):")
print(f"  RNA : min={z_rna.values.min():.2f}  max={z_rna.values.max():.2f}")
print(f"  CGN : min={z_cgn.values.min():.2f}  max={z_cgn.values.max():.2f}")
print(f"  CHN : min={z_chn.values.min():.2f}  max={z_chn.values.max():.2f}")

Z-score ranges (should be centred near 0):
  RNA : min=-3.82  max=4.24
  CGN : min=-4.24  max=4.24
  CHN : min=-4.24  max=4.24


## 6. Compute Weighted Discordance Score (WDS)
```
WDS = α·Z(CGN) + β·Z(CHN) - Z(RNA)
WDS >> 0 → FORCED_SUPPRESSION  (high methylation, low RNA)
WDS << 0 → FORCED_EXPRESSION   (low methylation, high RNA)
```

In [15]:
wds = ALPHA * z_cgn + BETA * z_chn - z_rna   # genes x cell_types

print(f"WDS matrix shape : {wds.shape}  (genes x cell types)")
print(f"WDS range        : {wds.values.min():.2f} to {wds.values.max():.2f}")
print(f"Discordance threshold: |WDS| >= {DS_THRESH}")
print(f"\nGenes with |WDS| >= {DS_THRESH} in Layer V ET ({LAYER_V_ET}):")
n_forced_on  = (wds[LAYER_V_ET] <= -DS_THRESH).sum()
n_forced_off = (wds[LAYER_V_ET] >=  DS_THRESH).sum()
print(f"  FORCED_EXPRESSION  (WDS <= -{DS_THRESH}): {n_forced_on} genes")
print(f"  FORCED_SUPPRESSION (WDS >=  {DS_THRESH}): {n_forced_off} genes")

WDS matrix shape : (30147, 19)  (genes x cell types)
WDS range        : -7.56 to 6.52
Discordance threshold: |WDS| >= 2.0

Genes with |WDS| >= 2.0 in Layer V ET (L5 ET):
  FORCED_EXPRESSION  (WDS <= -2.0): 2292 genes
  FORCED_SUPPRESSION (WDS >=  2.0): 969 genes


## 7. Build Per-Cell-Type Discordance Tables

In [16]:
def get_discordant_genes(wds_col, z_rna_col, z_cgn_col, z_chn_col, threshold, celltype_name):
    """
    For one cell type, return a DataFrame of discordant genes with all scores.
    """
    df = pd.DataFrame({
        "gene"       : wds_col.index,
        "WDS"        : wds_col.values,
        "Z_RNA"      : z_rna_col.values,
        "Z_CGN"      : z_cgn_col.values,
        "Z_CHN"      : z_chn_col.values,
        "cell_type"  : celltype_name,
    })
    df["category"] = "CONCORDANT"
    df.loc[df["WDS"] <= -threshold, "category"] = "FORCED_EXPRESSION"
    df.loc[df["WDS"] >=  threshold, "category"] = "FORCED_SUPPRESSION"
    df = df[df["category"] != "CONCORDANT"].copy()
    df = df.sort_values("WDS", key=abs, ascending=False).reset_index(drop=True)
    return df


discordant_per_ct = {}   # cell_type -> DataFrame of discordant genes

for ct in ALL_CELL_TYPES:
    df = get_discordant_genes(
        wds[ct], z_rna[ct], z_cgn[ct], z_chn[ct], DS_THRESH, ct
    )
    discordant_per_ct[ct] = df
    n_fe = (df["category"] == "FORCED_EXPRESSION").sum()
    n_fs = (df["category"] == "FORCED_SUPPRESSION").sum()
    print(f"  {ct:<30} FORCED_EXPRESSION={n_fe:>4}  FORCED_SUPPRESSION={n_fs:>4}")

# Save individual CSVs
print("\nSaving per-cell-type CSVs...")
for ct, df in discordant_per_ct.items():
    safe = ct.replace("/", "_").replace(" ", "_")
    path = OUT_DIR / f"discordant_{safe}.csv"
    df.to_csv(path, index=False)
print(f"Saved {len(discordant_per_ct)} files to {OUT_DIR}/")

  L5 ET                          FORCED_EXPRESSION=2292  FORCED_SUPPRESSION= 969
  Astro                          FORCED_EXPRESSION=2116  FORCED_SUPPRESSION=2770
  Macrophage                     FORCED_EXPRESSION=2464  FORCED_SUPPRESSION=4700
  Oligo                          FORCED_EXPRESSION=1880  FORCED_SUPPRESSION=2643
  OPC                            FORCED_EXPRESSION=2235  FORCED_SUPPRESSION=2334
  Endo                           FORCED_EXPRESSION=2152  FORCED_SUPPRESSION=3216
  VLMC                           FORCED_EXPRESSION=1747  FORCED_SUPPRESSION=4075
  SMC                            FORCED_EXPRESSION=2116  FORCED_SUPPRESSION=6241
  L2/3 IT                        FORCED_EXPRESSION=2782  FORCED_SUPPRESSION= 211
  L5 IT                          FORCED_EXPRESSION=1837  FORCED_SUPPRESSION= 100
  L5/6 NP                        FORCED_EXPRESSION=1842  FORCED_SUPPRESSION= 259
  L6 IT                          FORCED_EXPRESSION=1965  FORCED_SUPPRESSION= 158
  L6 CT                     

## 8. Stepwise Narrowing to Layer V ET-Specific Discordance

For each direction (FORCED_EXPRESSION / FORCED_SUPPRESSION) separately:
- **All cell types** share it → not identity-specific
- **Only neurons** share it → neuron-identity regulation
- **Only exitory neurons** share it → exitory-neuron-identity regulation
- **Only deep layer neurons** share it → deep layer regulation
- **Only Layer V ET** → Layer V ET-specific regulation

In [19]:
results = {}
for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
    print(f"\n{'='*60}")
    print(f"Category: {cat}")
    print('='*60)

    layerVET_genes       = gene_set(LAYER_V_ET, cat)
    non_neuron_genes     = union_sets(NON_NEURON_TYPES,       cat)
    inhibitory_genes     = union_sets(INHIBITORY_TYPES,       cat)
    upper_layer_genes    = union_sets(UPPER_LAYER_TYPES,      cat)
    deep_layer_genes     = union_sets(OTHER_DEEP_LAYER_TYPES, cat)

    # Tier 0: in L5 ET AND in non-neurons (truly non-specific)
    universal = layerVET_genes & non_neuron_genes

    # Tier 1: in L5 ET AND in some neuron, but NOT in non-neurons
    all_neuron_genes = inhibitory_genes | upper_layer_genes | deep_layer_genes
    neuron_specific = (layerVET_genes & all_neuron_genes) - non_neuron_genes

    # Tier 2: in L5 ET AND in excitatory neurons, NOT in inhibitory, NOT in non-neurons
    excitatory_genes = upper_layer_genes | deep_layer_genes
    excitatory_neuron_specific = (layerVET_genes & excitatory_genes) - inhibitory_genes - non_neuron_genes

    # Tier 3: in L5 ET AND in deep layer, NOT in upper layer, NOT in inhibitory, NOT in non-neurons
    deep_specific = (layerVET_genes & deep_layer_genes) - upper_layer_genes - inhibitory_genes - non_neuron_genes

    # Tier 4: ONLY in L5 ET — not in any other group
    all_other_genes = non_neuron_genes | inhibitory_genes | upper_layer_genes | deep_layer_genes
    layerVET_specific = layerVET_genes - all_other_genes

    print(f"  Layer V ET discordant genes total        : {len(layerVET_genes)}")
    print(f"  Tier 0 — Universal (non-specific)        : {len(universal)}")
    print(f"  Tier 1 — Neuron-identity specific        : {len(neuron_specific)}")
    print(f"  Tier 2 — Excitatory-neuron specific      : {len(excitatory_neuron_specific)}")
    print(f"  Tier 3 — Deep layer specific             : {len(deep_specific)}")
    print(f"  Tier 4 — Layer V ET specific             : {len(layerVET_specific)}")

    results[cat] = {
        "universal"                : universal,
        "neuron_specific"          : neuron_specific,
        "excitatory_neuron_specific": excitatory_neuron_specific,
        "deep_specific"            : deep_specific,
        "layerVET_specific"        : layerVET_specific,
    }


Category: FORCED_EXPRESSION
  Layer V ET discordant genes total        : 2292
  Tier 0 — Universal (non-specific)        : 289
  Tier 1 — Neuron-identity specific        : 1348
  Tier 2 — Excitatory-neuron specific      : 828
  Tier 3 — Deep layer specific             : 444
  Tier 4 — Layer V ET specific             : 655

Category: FORCED_SUPPRESSION
  Layer V ET discordant genes total        : 969
  Tier 0 — Universal (non-specific)        : 136
  Tier 1 — Neuron-identity specific        : 57
  Tier 2 — Excitatory-neuron specific      : 47
  Tier 3 — Deep layer specific             : 32
  Tier 4 — Layer V ET specific             : 776


## 9. Build Summary DataFrames with Full WDS Scores

In [23]:
# -- Gene symbol mapping (run against RNA; adjust adata_rna_use if needed) ---
print("var columns:", adata_rna_use.var.columns.tolist())
print(adata_rna_use.var.head(3))

SYMBOL_COL = None
for candidate in ['feature_name', 'gene_name', 'gene_symbol', 'symbol']:
    if candidate in adata_rna_use.var.columns:
        SYMBOL_COL = candidate
        break

if SYMBOL_COL:
    id_to_symbol = adata_rna_use.var[SYMBOL_COL].to_dict()
    print(f"Using '{SYMBOL_COL}' for gene symbols")
else:
    id_to_symbol = {g: g for g in adata_rna_use.var_names}
    print("No symbol column found — using var_names as gene symbols")

print(f"Loaded {len(id_to_symbol)} gene symbols — example: {list(id_to_symbol.items())[:3]}")
# ---------------------------------------------------------------------------


def build_wds_summary(gene_set_input, tier_label, category):
    """
    For each gene, collect WDS, Z_RNA, Z_CGN, Z_CHN across all cell types.
    Adds a 'gene_symbol' column resolved from id_to_symbol.
    """
    if not gene_set_input:
        return pd.DataFrame()

    rows = []
    for gene in sorted(gene_set_input):
        row = {
            "gene"        : gene,
            "gene_symbol" : id_to_symbol.get(gene, gene),   # ← symbol added here
            "tier"        : tier_label,
            "category"    : category,
        }
        for ct in ALL_CELL_TYPES:
            if gene in wds.index:
                row[f"{ct}__WDS"]   = round(wds.loc[gene, ct],   3)
                row[f"{ct}__Z_RNA"] = round(z_rna.loc[gene, ct], 3)
                row[f"{ct}__Z_CGN"] = round(z_cgn.loc[gene, ct], 3)
                row[f"{ct}__Z_CHN"] = round(z_chn.loc[gene, ct], 3)
        rows.append(row)

    df = pd.DataFrame(rows)

    # Reorder: gene | gene_symbol | tier | category | ... scores ...
    front_cols = ["gene", "gene_symbol", "tier", "category"]
    df = df[front_cols + [c for c in df.columns if c not in front_cols]]

    wds_cols = [c for c in df.columns if c.endswith("__WDS")]
    df["mean_WDS_all_ct"] = df[wds_cols].mean(axis=1).round(3)
    df = df.sort_values("mean_WDS_all_ct", key=abs, ascending=False).reset_index(drop=True)
    return df


tier_labels = {
    "universal"                  : "Tier0_Universal",
    "neuron_specific"            : "Tier1_Neuron_Specific",
    "excitatory_neuron_specific" : "Tier2_Excitatory_Neuron_Specific",
    "deep_specific"              : "Tier3_DeepLayer_Specific",
    "layerVET_specific"          : "Tier4_LayerVET_Specific",
}

all_summaries = {}
for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
    for tier_key, tier_label in tier_labels.items():
        gene_s = results[cat][tier_key]
        df     = build_wds_summary(gene_s, tier_label, cat)
        key    = f"{cat}__{tier_label}"
        all_summaries[key] = df

print("Summary DataFrames built.")
for k, v in all_summaries.items():
    print(f"  {k}: {len(v)} genes")

var columns: ['feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
                   feature_name feature_reference feature_biotype  \
ENSMUSG00000029422        Rsrc2   NCBITaxon:10090            gene   
ENSMUSG00000114536      Gm48837   NCBITaxon:10090            gene   
ENSMUSG00000049036      Tmem121   NCBITaxon:10090            gene   

                   feature_length    feature_type  
ENSMUSG00000029422           1377  protein_coding  
ENSMUSG00000114536           2849          lncRNA  
ENSMUSG00000049036           1574  protein_coding  
Using 'feature_name' for gene symbols
Loaded 30198 gene symbols — example: [('ENSMUSG00000029422', 'Rsrc2'), ('ENSMUSG00000114536', 'Gm48837'), ('ENSMUSG00000049036', 'Tmem121')]
Summary DataFrames built.
  FORCED_EXPRESSION__Tier0_Universal: 289 genes
  FORCED_EXPRESSION__Tier1_Neuron_Specific: 1348 genes
  FORCED_EXPRESSION__Tier2_Excitatory_Neuron_Specific: 828 genes
  FORCED_EXPRESSION__Tier3_DeepLayer_S

## 10. Save All Results to CSV

In [24]:
# Save individual tier/category files
for key, df in all_summaries.items():
    if df.empty:
        print(f"[SKIP] {key} — empty")
        continue
    path = OUT_DIR / f"{key}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path.name}  ({len(df)} genes)")

# Save one master file combining everything
master = pd.concat([df for df in all_summaries.values() if not df.empty], ignore_index=True)
master_path = OUT_DIR / "MASTER_discordance_summary.csv"
master.to_csv(master_path, index=False)
print(f"\nMaster file saved: {master_path.name}  ({len(master)} genes total)")

Saved: FORCED_EXPRESSION__Tier0_Universal.csv  (289 genes)
Saved: FORCED_EXPRESSION__Tier1_Neuron_Specific.csv  (1348 genes)
Saved: FORCED_EXPRESSION__Tier2_Excitatory_Neuron_Specific.csv  (828 genes)
Saved: FORCED_EXPRESSION__Tier3_DeepLayer_Specific.csv  (444 genes)
Saved: FORCED_EXPRESSION__Tier4_LayerVET_Specific.csv  (655 genes)
Saved: FORCED_SUPPRESSION__Tier0_Universal.csv  (136 genes)
Saved: FORCED_SUPPRESSION__Tier1_Neuron_Specific.csv  (57 genes)
Saved: FORCED_SUPPRESSION__Tier2_Excitatory_Neuron_Specific.csv  (47 genes)
Saved: FORCED_SUPPRESSION__Tier3_DeepLayer_Specific.csv  (32 genes)
Saved: FORCED_SUPPRESSION__Tier4_LayerVET_Specific.csv  (776 genes)

Master file saved: MASTER_discordance_summary.csv  (4612 genes total)


## 11. Preview: Layer V ET-Specific Genes

In [26]:
preview_cols = ["gene", "gene_symbol", "category", "tier", "mean_WDS_all_ct",
                f"{LAYER_V_ET}__WDS", f"{LAYER_V_ET}__Z_RNA",
                f"{LAYER_V_ET}__Z_CGN", f"{LAYER_V_ET}__Z_CHN"]

for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
    key = f"{cat}__Tier4_LayerVET_Specific"
    df  = all_summaries.get(key, pd.DataFrame())
    print(f"\n{'='*60}")
    print(f"Layer V ET-Specific {cat} (top 20)")
    print('='*60)
    if df.empty:
        print("No genes found — consider lowering DS_THRESH in Cell 0")
    else:
        available = [c for c in preview_cols if c in df.columns]
        display(df[available].head(20))


Layer V ET-Specific FORCED_EXPRESSION (top 20)


,gene,gene_symbol,category,tier,mean_WDS_all_ct,L5 ET__WDS,L5 ET__Z_RNA,L5 ET__Z_CGN,L5 ET__Z_CHN
0,ENSMUSG00000116989,Gm49709,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-4.931,3.897,-1.636,-0.131
1,ENSMUSG00000000103,Zfy2,FORCED_EXPRESSION,Tier4_LayerVET_Specific,0.0,-2.566,2.886,0.988,-0.681
2,ENSMUSG00000000308,Ckmt1,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-4.485,2.134,-3.174,-1.116
3,ENSMUSG00000000759,Tubgcp3,FORCED_EXPRESSION,Tier4_LayerVET_Specific,0.0,-2.698,1.175,-1.923,-0.924
4,ENSMUSG00000000817,Fasl,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-3.168,2.925,-0.810,0.608
5,ENSMUSG00000000916,Nsun5,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-2.796,1.674,-1.213,-0.984
6,ENSMUSG00000001909,Trmt1,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-2.182,1.184,-1.057,-0.907
7,ENSMUSG00000002010,Idh3g,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-2.694,2.032,-0.432,-1.009
8,ENSMUSG00000002833,Hdgfl2,FORCED_EXPRESSION,Tier4_LayerVET_Specific,-0.0,-2.220,1.419,-0.737,-0.896
9,ENSMUSG00000002949,Timm44,FORCED_EXPRESSION,Tier4_LayerVET_Specific,0.0,-2.108,1.040,-1.183,-0.897



Layer V ET-Specific FORCED_SUPPRESSION (top 20)


,gene,gene_symbol,category,tier,mean_WDS_all_ct,L5 ET__WDS,L5 ET__Z_RNA,L5 ET__Z_CGN,L5 ET__Z_CHN
0,ENSMUSG00000116983,4930553J12Rik,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,0.0,2.004,-0.529,1.219,1.858
1,ENSMUSG00000000003,Pbsn,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,0.0,2.183,-0.395,1.287,2.540
2,ENSMUSG00000000204,Slfn4,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,0.0,2.087,-0.523,1.518,1.633
3,ENSMUSG00000000320,Alox12,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,0.0,2.067,-0.254,1.919,1.654
4,ENSMUSG00000000673,Haao,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,0.0,2.087,-0.979,1.208,0.958
5,ENSMUSG00000000693,Loxl3,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,-0.0,2.341,-0.515,1.947,1.644
6,ENSMUSG00000001225,Slc26a3,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,0.0,2.144,-0.694,1.449,1.452
7,ENSMUSG00000001642,Akr1b1,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,-0.0,2.211,-0.868,1.139,1.649
8,ENSMUSG00000111482,Gm47153,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,-0.0,2.282,-0.442,2.059,1.510
9,ENSMUSG00000111517,Or4a39,FORCED_SUPPRESSION,Tier4_LayerVET_Specific,-0.0,2.173,-0.787,0.997,1.970


## 12. Threshold Sensitivity Check

In [27]:
print(f"{'DS_thresh':>10}  {'FE_Tier3':>10}  {'FS_Tier3':>10}  {'FE_Tier1':>10}  {'FS_Tier1':>10}")
print("-" * 55)

for thresh in [1.0, 1.5, 2.0, 2.5, 3.0]:
    tmp = {}
    for ct in ALL_CELL_TYPES:
        for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
            col = wds[ct]
            if cat == "FORCED_EXPRESSION":
                genes = set(col[col <= -thresh].index)
            else:
                genes = set(col[col >=  thresh].index)
            tmp[(ct, cat)] = genes

    def gs(ct, cat): return tmp.get((ct, cat), set())
    def u(cts, cat): return set.union(*[gs(ct, cat) for ct in cts]) if cts else set()

    for cat, sign in [("FORCED_EXPRESSION", "FE"), ("FORCED_SUPPRESSION", "FS")]:
        lv   = gs(LAYER_V_ET, cat)
        non  = u(NON_NEURON_TYPES, cat)
        exi  = u(INHIBITORY_TYPES, cat)
        upl  = u(UPPER_LAYER_TYPES, cat)
        dep  = u(OTHER_DEEP_LAYER_TYPES, cat)
        tmp[f"{sign}_t3_{thresh}"] = len(lv - non - upl - dep)
        tmp[f"{sign}_t1_{thresh}"] = len((lv | upl | dep) - non)

    print(f"{thresh:>10.1f}  "
          f"{tmp['FE_t3_' + str(thresh)]:>10}  "
          f"{tmp['FS_t3_' + str(thresh)]:>10}  "
          f"{tmp['FE_t1_' + str(thresh)]:>10}  "
          f"{tmp['FS_t1_' + str(thresh)]:>10}")

print("\nFE=FORCED_EXPRESSION, FS=FORCED_SUPPRESSION, Tier4=LayerVET-specific, Tier1=Neuron-specific")

 DS_thresh    FE_Tier3    FS_Tier3    FE_Tier1    FS_Tier1
-------------------------------------------------------
       1.0         361        1071        8355        5799
       1.5         834        2008        9272        4032
       2.0        1021         786        8085        1461
       2.5         844         182        6145         386
       3.0         557          24        4311          80

FE=FORCED_EXPRESSION, FS=FORCED_SUPPRESSION, Tier4=LayerVET-specific, Tier1=Neuron-specific


## 13. Intersect with RNA DE Results (Optional)
Cross-reference with Layer V ET-specific genes from the RNA notebook for highest-confidence candidates.

In [33]:
# Forced Expression
RNA_DE_PATH = "/home/nakagawa/datasets/LayerV_ET_results_rnaseq/LayerVET_specific_genes.csv"
try:
    rna_de = pd.read_csv(RNA_DE_PATH)
    rna_de_genes = set(rna_de["gene"])
    print(f"Loaded {len(rna_de_genes)} Layer V ET-specific genes from RNA DE analysis")

    for cat in ["FORCED_EXPRESSION"]:
        key = f"{cat}__Tier4_LayerVET_Specific"   # ← was Tier3, fixed to match your tier labels
        df  = all_summaries.get(key, pd.DataFrame())
        if df.empty:
            continue

        overlap = set(df["gene"]) & rna_de_genes
        print(f"\n{cat} Tier4 x RNA DE overlap: {len(overlap)} genes")
        print("  → These are your TOP PRIORITY candidates (discordant + DE specific)")

        overlap_df = df[df["gene"].isin(overlap)].copy()

        # Preview table
        preview_cols = ["gene", "gene_symbol", "mean_WDS_all_ct",
                        f"{LAYER_V_ET}__WDS", f"{LAYER_V_ET}__Z_RNA",
                        f"{LAYER_V_ET}__Z_CGN", f"{LAYER_V_ET}__Z_CHN"]
        available = [c for c in preview_cols if c in overlap_df.columns]
        display(overlap_df[available]
                .sort_values(f"{LAYER_V_ET}__WDS", key=abs, ascending=False)
                .head(20)
                .reset_index(drop=True))

        overlap_df.to_csv(OUT_DIR / f"TOP_PRIORITY_{cat}.csv", index=False)

except FileNotFoundError:
    print("RNA DE results not found — run RNA notebook first, or update RNA_DE_PATH above")

Loaded 65 Layer V ET-specific genes from RNA DE analysis

FORCED_EXPRESSION Tier4 x RNA DE overlap: 18 genes
  → These are your TOP PRIORITY candidates (discordant + DE specific)


,gene,gene_symbol,mean_WDS_all_ct,L5 ET__WDS,L5 ET__Z_RNA,L5 ET__Z_CGN,L5 ET__Z_CHN
0,ENSMUSG00000022206,Npr3,-0.0,-7.068,4.092,-3.467,-2.240
1,ENSMUSG00000087057,Gm11730,0.0,-6.778,4.204,-2.977,-1.970
2,ENSMUSG00000106515,Gm30382,-0.0,-5.909,2.848,-3.678,-2.136
3,ENSMUSG00000039620,Trmt9b,-0.0,-5.574,2.771,-3.589,-1.626
4,ENSMUSG00000022843,Clcn2,-0.0,-5.316,3.420,-2.355,-1.208
5,ENSMUSG00000059857,Ntng1,-0.0,-4.843,3.259,-1.736,-1.356
6,ENSMUSG00000020140,Lgr5,0.0,-4.834,3.805,-1.286,-0.644
7,ENSMUSG00000045441,Gprin3,-0.0,-4.643,2.527,-2.538,-1.483
8,ENSMUSG00000041565,L3mbtl4,-0.0,-4.324,3.776,-0.699,-0.322
9,ENSMUSG00000059852,Kcng2,-0.0,-4.072,3.315,-0.732,-0.795


In [32]:
# Forced Suppression
RNA_DE_PATH = "/home/nakagawa/datasets/LayerV_ET_results_rnaseq/LayerVET_specific_suppressed_genes.csv"
try:
    rna_de = pd.read_csv(RNA_DE_PATH)
    rna_de_genes = set(rna_de["gene"])
    print(f"Loaded {len(rna_de_genes)} Layer V ET-specific genes from RNA DE analysis")

    for cat in ["FORCED_SUPPRESSION"]:
        key = f"{cat}__Tier4_LayerVET_Specific"   # ← was Tier3, fixed to match your tier labels
        df  = all_summaries.get(key, pd.DataFrame())
        if df.empty:
            continue

        overlap = set(df["gene"]) & rna_de_genes
        print(f"\n{cat} Tier4 x RNA DE overlap: {len(overlap)} genes")
        print("  → These are your TOP PRIORITY candidates (discordant + DE specific)")

        overlap_df = df[df["gene"].isin(overlap)].copy()

        # Preview table
        preview_cols = ["gene", "gene_symbol", "mean_WDS_all_ct",
                        f"{LAYER_V_ET}__WDS", f"{LAYER_V_ET}__Z_RNA",
                        f"{LAYER_V_ET}__Z_CGN", f"{LAYER_V_ET}__Z_CHN"]
        available = [c for c in preview_cols if c in overlap_df.columns]
        display(overlap_df[available]
                .sort_values(f"{LAYER_V_ET}__WDS", key=abs, ascending=False)
                .head(20)
                .reset_index(drop=True))

        overlap_df.to_csv(OUT_DIR / f"TOP_PRIORITY_{cat}.csv", index=False)

except FileNotFoundError:
    print("RNA DE results not found — run RNA notebook first, or update RNA_DE_PATH above")

Loaded 16 Layer V ET-specific genes from RNA DE analysis

FORCED_SUPPRESSION Tier4 x RNA DE overlap: 2 genes
  → These are your TOP PRIORITY candidates (discordant + DE specific)


,gene,gene_symbol,mean_WDS_all_ct,L5 ET__WDS,L5 ET__Z_RNA,L5 ET__Z_CGN,L5 ET__Z_CHN
0,ENSMUSG00000052920,Prkg1,-0.0,2.434,-0.972,1.762,1.015
1,ENSMUSG00000030307,Slc6a11,-0.0,2.123,-0.523,1.288,2.069
